In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict
from pathlib import Path
import os

In [7]:
load_dotenv(Path(r"d:\DATA D DRIVE\LangGraph") / ".env")
print("HF token loaded:", bool(os.getenv("HUGGINGFACEHUB_API_TOKEN")))

HF token loaded: True


In [8]:
# HuggingFaceEndpoint uses Inference Providers. Qwen2.5-0.5B-Instruct is not
# served by any provider enabled on this Hugging Face account (only Featherless
# lists it). Use a provider-hosted Qwen chat model instead.
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    provider="novita",
)
model = ChatHuggingFace(llm=llm)


d:\DATA D DRIVE\LangGraph\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
#create a state
class LLMState(TypedDict):
    question:str
    answer:str

In [12]:
def llm_qa(State:LLMState)->LLMState:
    question=State["question"]
    prompt=f"Answer the following question:{question}"
    answer=model.invoke(prompt).content
    State["answer"]=answer
    return State

In [13]:
#create a graph
graph = StateGraph(LLMState)

#create a node
graph.add_node("llm_qa",llm_qa)
graph.add_edge(START,"llm_qa")
graph.add_edge("llm_qa",END)

workflow=graph.compile()


In [15]:
#initialize the graph
initial_state={"question":"capital of france?"}
final_state=workflow.invoke(initial_state)
print(final_state)

BadRequestError: (Request ID: Root=1-6a9afd27-62f15e421ed2d3810c81cfbb;6b367b9c-8fe6-447e-9bcf-a576d8abff4f)

Bad request:
{'message': "The requested model 'Qwen/Qwen2.5-0.5B-Instruct' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}